# ISIC skin lesion · Baby Dragon Hatchling · Hebbian explainability bundle

Trains a **dermoscopy skin-lesion classifier** on **ISIC / HAM10000** with the
`hatchvision` framework — the *same* pipeline as the CUB-200 notebook, only the
dataset loader changes (`DATASET = "isic"`). It records **Hebbian co-activation**
of the BDH sparse-neuron space during training, clusters neurons into concepts,
and exports the full **in-browser inference bundle** for the web app:

- `graph.json` — the Hebbian concept graph (IVGraph)
- `model.onnx` — the classifier with neuron-activation outputs
- `manifest.json` — preprocessing + node mapping
- `explain.json` — per-class activation regions + exact SHAP influence
- `hebbian_state.pt` — raw statistics for post-hoc graph rebuilds

Skin lesions have **no per-image visual-attribute annotations** (unlike CUB's
312 attributes), so concepts are **grounded by the classes they respond to**
instead of by named attributes — the framework falls back automatically. Class
**activation regions**, **SHAP influence** and the **Neurons** network view all
work identically.

**How to run this on Kaggle**
1. *Add Input* → attach a skin-lesion dataset. Recommended:
   **`kmader/skin-cancer-mnist-ham10000`** (HAM10000 — 10 015 dermoscopy images,
   7 diagnoses). ISIC-2019 one-hot ground-truth CSVs and folder-per-class
   datasets (e.g. `nodoubttome/skin-cancer9-classesisic`) are auto-detected too.
2. *Settings* → **Accelerator: GPU** (T4/P100), **Internet: On** (to clone the
   repo and fetch pretrained weights).
3. *Run All* (~15–25 min on a T4).
4. Download **`/kaggle/working/bundle.zip`** and **drag it straight onto the
   deployed web app** — it loads in the browser, no redeploy needed.

> ⚠️ **Research / education only** — not a diagnostic device; the explanations
> describe what the model learned, not clinical ground truth. HAM10000 has
> several images per lesion, so the loader splits **by `lesion_id`** to avoid
> train/val leakage that would inflate accuracy.

In [ ]:
# Install hatchvision from the repo
BRANCH = "main"
!git clone --depth 1 -b {BRANCH} https://github.com/LarsGroep/DragonHatchling.git
%cd DragonHatchling
!pip install -q onnx onnxruntime onnxconverter-common

import sys, torch
sys.path.insert(0, ".")
import hatchvision
print("hatchvision", hatchvision.__version__, "· torch", torch.__version__,
      "· cuda:", torch.cuda.is_available())

In [ ]:
# ---------------------------- configuration ----------------------------
DATASET      = "isic"      # registered skin-lesion loader (CSV or folders, auto-detected)
BACKBONE     = "hybrid"    # "hybrid" (pretrained encoder + BDH neurons)  or  "bdh" (pure)
EPOCHS       = 14
BATCH_SIZE   = 64
LR           = 1e-3
MAX_UNITS    = 512         # Hebbian tracked neurons (subsampled from neuron_dim)
N_CONCEPTS   = 16          # fewer classes than CUB -> fewer concepts read better
PROBE_IMAGES = 768         # val images used for concept exemplars + SHAP background
VAL_FRAC     = 0.15        # used when the dataset ships no explicit test split
BALANCE      = True        # class-balanced sampler (HAM10000 is ~67% nevus)

BACKBONE_KWARGS = {
    "hybrid": dict(encoder="resnet50", pretrained=True, freeze_encoder=True,
                   neuron_dim=4096),
    "bdh":    dict(patch_size=16, dim=256, neuron_dim=2048, depth=6,
                   share_weights=True),
}[BACKBONE]
IMAGE_SIZE = 224 if BACKBONE == "hybrid" else 128
if BACKBONE == "bdh":
    EPOCHS, LR = max(EPOCHS, 30), 3e-4

In [ ]:
# ------------------------- locate the ISIC data -------------------------
from pathlib import Path

def find_isic_root():
    base = Path("/kaggle/input")
    # 1) a metadata / ground-truth CSV (HAM10000 or ISIC one-hot)
    for c in sorted(base.rglob("*.csv")):
        try:
            head = c.open().readline().lower()
        except Exception:
            continue
        cols = [h.strip() for h in head.split(",")]
        if "dx" in cols or sum(k in cols for k in ("mel", "nv", "bcc")) >= 2:
            return c.parent
    # 2) a train/<class>/ folder tree
    for d in sorted(base.rglob("*")):
        if d.is_dir() and d.name.lower() in {"train", "training"} \
                and any(x.is_dir() for x in d.iterdir()):
            return d.parent
    return base

root = find_isic_root()
print("ISIC root:", root)

In [ ]:
from collections import Counter
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from hatchvision import (HebbianFeatureMemory, TrainConfig, Trainer,
                         build_loader, create_model)

data = build_loader(DATASET, root=str(root), image_size=IMAGE_SIZE, val_frac=VAL_FRAC)
train_ds, val_ds = data.train_dataset(), data.val_dataset()
dist = Counter(train_ds.targets)
print(data.spec.num_classes, "classes ·", len(train_ds), "train /", len(val_ds), "val")
for i, name in enumerate(data.spec.class_names):
    print(f"  {name:<26} {dist.get(i, 0):>5} train images")

# class-balanced sampling so minority diagnoses aren't drowned out by nevi
if BALANCE:
    counts = torch.tensor([dist.get(i, 0) for i in range(data.spec.num_classes)], dtype=torch.float)
    weights = (1.0 / counts.clamp(min=1))[torch.tensor(train_ds.targets)]
    sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=2)

model = create_model(BACKBONE, data.spec, **BACKBONE_KWARGS)
memory = HebbianFeatureMemory(model, num_classes=data.spec.num_classes, max_units=MAX_UNITS)
print("Hebbian memory observes:", list(model.hebbian_layers()),
      "· class-balanced sampler:", BALANCE)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_train/1e6:.1f} M")

In [ ]:
trainer = Trainer(model, TrainConfig(epochs=EPOCHS, lr=LR, log_every=25), memory)
history = trainer.fit(train_loader, val_loader)
print(f"final val accuracy: {history['val_acc'][-1]:.3f}")

## Balanced view: per-class recall

Overall accuracy is misleading on imbalanced skin data (predicting "nevus" for
everything already scores high). Per-class recall + the macro-average show
whether the minority diagnoses are actually being learned.

In [ ]:
from collections import defaultdict
import torch

model.eval()
dev = next(model.parameters()).device
correct, total = defaultdict(int), defaultdict(int)
with torch.no_grad():
    for xb, yb in val_loader:
        pred = model(xb.to(dev)).argmax(1).cpu()
        for p, t in zip(pred, yb):
            total[int(t)] += 1
            correct[int(t)] += int(p == t)

recalls = []
for i, name in enumerate(data.spec.class_names):
    r = correct[i] / total[i] if total[i] else 0.0
    recalls.append(r)
    print(f"  {name:<26} recall {r:6.1%}  (n={total[i]})")
print(f"\nbalanced accuracy (mean recall): {sum(recalls)/len(recalls):.1%}")

## Concepts: cluster the Hebbian co-activation

Units that fired together during training are clustered into concepts. ISIC has
no per-image attribute file, so each concept is labelled by the **diagnoses it
responds to** (class affinity) plus its exemplar images — the same clustering
that names CUB concepts by attributes, minus the attribute grounding step.

In [ ]:
from hatchvision.explain import cluster_concepts, find_exemplars, ground_concepts

layer = memory.layer_names[-1]
concepts = cluster_concepts(memory, layer, data.spec.class_names, n_concepts=N_CONCEPTS)
probe = data.probe_batch(PROBE_IMAGES)
find_exemplars(concepts, memory, model, probe)

# attribute grounding runs only if the dataset provides attributes (ISIC does not)
attr_names = data.attribute_names()
attr_matrix = data.probe_attributes(probe.shape[0])
if attr_names and attr_matrix is not None:
    ground_concepts(concepts, memory, model, probe, attr_matrix, attr_names)
    print(f"grounded {sum(1 for c in concepts if c.attributes)}/{len(concepts)} concepts")
else:
    print("no per-image attributes for ISIC — concepts grounded by class affinity")

for c in concepts[:N_CONCEPTS]:
    top = sorted(c.class_affinity.items(), key=lambda kv: -kv[1])[:2] if c.class_affinity else []
    responds = " / ".join(n for n, _ in top) or "—"
    print(f"[{c.concept_id:>2}] {len(c.units):>3} units  coh {c.coherence:.2f}  responds to: {responds}")

In [ ]:
# visual check: exemplar dermoscopy images for the strongest concepts
import matplotlib.pyplot as plt
from hatchvision.explain import denormalize

mean, std = data.spec.normalization()
show = [c for c in concepts if c.exemplars][:6]
fig, axes = plt.subplots(len(show), 6, figsize=(14, 2.4 * len(show)))
for row, c in zip(axes, show):
    for ax, idx in zip(row, c.exemplars):
        img = denormalize(probe[idx:idx + 1], mean, std)[0]
        ax.imshow(img.permute(1, 2, 0))
        ax.axis("off")
    row[0].set_title(c.label, loc="left", fontsize=9)
plt.tight_layout(); plt.show()

## Export the web-app bundle

In [ ]:
from hatchvision.export import export_ivgraph, export_onnx_bundle, export_explain_pack

OUT = Path("/kaggle/working/bundle")
export_ivgraph(memory, concepts, layer, data.spec.class_names, OUT / "graph.json",
               meta={"dataset": data.spec.name, "backbone": BACKBONE,
                     "val_acc": round(history["val_acc"][-1], 4)})
# demo explain pack: per-class Hebbian activation regions + the unit->class
# SHAP influence matrix (exact for the hybrid/bdh linear neuron readout) --
# powers the web app's class-region picker, SHAP bars and demo tour.
export_explain_pack(memory, layer, data.spec.class_names, OUT / "explain.json",
                    model=model, background=probe[:64])
# fp16 halves the in-browser download; predictions are unchanged
export_onnx_bundle(model.cpu(), memory, data.spec, OUT, fp16=True,
                   explain_file="explain.json",
                   extra_meta={"backbone": BACKBONE})
# raw Hebbian statistics: re-cluster / re-export later without retraining
torch.save(memory.state_dict(), OUT / "hebbian_state.pt")

import shutil
shutil.make_archive("/kaggle/working/bundle", "zip", OUT)
!ls -la /kaggle/working/bundle /kaggle/working/bundle.zip
print("""
Done. Download bundle.zip (right panel -> Output) and **drag it straight onto
the deployed web app** -- it loads in the browser (stored locally, survives
reloads) with no redeploy. Upload a dermoscopy image and you get top-k
diagnoses, the concepts that fired, exact SHAP influence per activation region,
and the class-region / neuron-network views. (ISIC has no visual-attribute
annotations, so the app's "visual evidence" attribute bars stay empty --
everything else works.)

To make it the site default instead: unzip into the repo's webapp/ and redeploy
    cd webapp && npx vercel deploy --prod

To iterate on clustering WITHOUT retraining:
    python scripts/rebuild_graph.py --state hebbian_state.pt \\
        --manifest manifest.json --dataset isic --root <isic_root> \\
        --n-concepts 16 --out graph.json --explain-out explain.json
""")

In [ ]:
# optional sanity check: ONNX output matches PyTorch
import onnxruntime, numpy as np, torch
sess = onnxruntime.InferenceSession(str(OUT / "model.onnx"))
x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
model.eval()
with torch.no_grad():
    ref = model(x).numpy()
out = sess.run(None, {"images": x.numpy()})[0]
print("max |onnx - torch| =", float(np.abs(out - ref).max()))